In [3]:
%pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-macosx_10_15_x86_64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-macosx_10_15_x86_64.whl (1.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data import load_raw_dataset
from src.features import build_feature_pipeline
from src.models import train_lgb_time_series

In [2]:
# Load sample dataset (e.g., 100,000 rows for fast benchmarking)
df_raw = load_raw_dataset(data_dir="../data/raw", split="train", nrows=100000)

# Generate upgraded feature matrix (includes ratios, email parsing, null counts)
df_fe = build_feature_pipeline(df_raw)

print(f"\nUpgraded feature matrix shape: {df_fe.shape}")

Loading train_transaction.csv...
Loading train_identity.csv...
Performing left join on TransactionID...
Optimizing memory footprint...
Memory usage decreased to 176.91 MB (46.6% reduction)
Starting feature engineering pipeline...
Feature engineering complete. Total columns: 453

Upgraded feature matrix shape: (100000, 453)


In [3]:
# 1. Define columns to drop
drop_cols = ["TransactionID", "TransactionDT", "isFraud", "uid1", "uid2"]

# 2. Run 5-Fold Time-Series CV
models, oof_preds, upgraded_auc = train_lgb_time_series(
    df=df_fe,
    target_col="isFraud",
    drop_cols=drop_cols,
    n_splits=5
)

# 3. Store baseline for comparison (replace 0.88819 with your original score if different)
baseline_auc = 0.88819
auc_diff = upgraded_auc - baseline_auc

# 4. Print Score Comparison Summary
print("=" * 45)
print("       CV SCORE COMPARISON SUMMARY           ")
print("=" * 45)
print(f"Baseline Mean CV ROC-AUC  : {baseline_auc:.5f}")
print(f"Upgraded Mean CV ROC-AUC  : {upgraded_auc:.5f}")
print(f"Net Score Delta           : {auc_diff:+.5f} {'🟢 (Improved)' if auc_diff > 0 else '🔴 (Regression)'}")
print("=" * 45)

# Save OOF predictions to DataFrame for Notebook 04 evaluation
df_fe["oof_preds"] = oof_preds

Starting Time-Series CV (5 Folds) on 448 features...


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 ROC-AUC: 0.87231


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 ROC-AUC: 0.92401


...

Fold 3 ROC-AUC: 0.90849


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 ROC-AUC: 0.91385


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 ROC-AUC: 0.90802

--- Mean Out-Of-Fold ROC-AUC: 0.90534 ---

       CV SCORE COMPARISON SUMMARY           
Baseline Mean CV ROC-AUC  : 0.88819
Upgraded Mean CV ROC-AUC  : 0.90534
Net Score Delta           : +0.01715 🟢 (Improved)
